# 3D SwinMamba + IS-MIM Module-Wise Output Walkthrough
### BraTS 2023 Adult Glioma Dataset Architecture Verification

This notebook provides a step-by-step module output breakdown of the **3D SwinMamba + IS-MIM** pipeline.
Each tensor shape, value statistics (min, mean, max, std), and semantic functionality is printed as features pass through the network.

In [1]:
import sys
import torch
import numpy as np

W_NAME  = 46
W_SHAPE = 30
W_STATS = 36
_DIVIDER = "=" * (W_NAME + W_SHAPE + W_STATS + 10)
_SEP     = "-" * (W_NAME + W_SHAPE + W_STATS + 10)

def _stats(t: torch.Tensor) -> str:
    f = t.detach().float()
    return f"min={f.min():+.3f}  u={f.mean():+.3f}  max={f.max():+.3f}  std={f.std():.3f}"

def _shape(t: torch.Tensor) -> str:
    return str(tuple(t.shape))

def log(label: str, tensor: torch.Tensor, note: str = ""):
    shape_str = _shape(tensor)
    stats_str = _stats(tensor)
    note_str  = f"  <- {note}" if note else ""
    print(f"  {label:<{W_NAME}}  {shape_str:<{W_SHAPE}}  {stats_str}{note_str}")

def section(title: str):
    print(f"\n{_DIVIDER}")
    print(f"  {title}")
    print(_DIVIDER)

def subsec(title: str):
    print(f"\n  {_SEP}")
    print(f"  {title}")
    print(f"  {_SEP}")

device = torch.device("cpu")
torch.manual_seed(42)
print(_DIVIDER)
print("  SwinMamba + IS-MIM  │  Module-Wise Output Verification")
print("  (B=1, 4-modality MRI, 96³ volume, float32, CPU)")
print(_DIVIDER)

  SwinMamba + IS-MIM  │  Module-Wise Output Verification
  (B=1, 4-modality MRI, 96³ volume, float32, CPU)


In [2]:
# Stage 0 — Raw Input Volume
section("STAGE 0 │ Raw 4-Modality MRI Input")
raw_volume = torch.randn(1, 4, 96, 96, 96)
log("Raw input  [B, C, D, H, W]", raw_volume, "T1 | T1ce | T2 | FLAIR stacked on channel dim")


  STAGE 0 │ Raw 4-Modality MRI Input
  Raw input  [B, C, D, H, W]                      (1, 4, 96, 96, 96)              min=-4.866  u=-0.001  max=+4.830  std=1.000  <- T1 | T1ce | T2 | FLAIR stacked on channel dim


In [3]:
# Stage 1 — IS-MIM Preprocessing (Self-Supervised Masking)
section("STAGE 1 │ IS-MIM Preprocessing (Pre-training Path)")
from src.data.is_mim_transforms import (
    image_to_tokens_3d,
    synthesize_modalities,
    apply_block_masking_3d,
    tokens_to_image_3d,
)

subsec("1a. 3D Patch Tokenisation (patch_size=16 → 6×6×6 = 216 tokens)")
tokens = image_to_tokens_3d(raw_volume, patch_size=16)
log("Tokens  [B, C, p, p, p, N_patches]", tokens, "each 16³ cuboid is one token, N_patches=216")

subsec("1b. Intra-Modality Patch Swap (swap_prob=0.5, 2 iterations)")
swapped_tokens, swap_mask = synthesize_modalities(tokens, swap_prob=0.5, num_iterations=2)
log("Swapped tokens  [B, C, p, p, p, N]", swapped_tokens, "patches exchanged across modality channels")
log("Swap mask  [B, N_patches]", swap_mask, "1 = patch swapped | 0 = unchanged")

subsec("1c. 50% Block-Wise Token Masking")
masked_tokens, is_masked = apply_block_masking_3d(swapped_tokens, mask_ratio=0.5)
log("Masked tokens  [B, C, p, p, p, N]", masked_tokens, "108/216 tokens zeroed out (50%)")
log("is_masked  [B, 1, 1, 1, 1, N]", is_masked, "1 = masked (loss computed here) | 0 = kept")

subsec("1d. Detokenise → Perturbed Volume")
D, H, W = 96, 96, 96
x_perturbed = tokens_to_image_3d(masked_tokens, orig_shape=(D, H, W), patch_size=16)
x_clean     = raw_volume.clone()
log("x_perturbed  [B, C, D, H, W]", x_perturbed, "corrupted volume → input to SwinMamba")
log("x_clean      [B, C, D, H, W]", x_clean, "original volume → reconstruction target")


  STAGE 1 │ IS-MIM Preprocessing (Pre-training Path)

  --------------------------------------------------------------------------------------------------------------------------
  1a. 3D Patch Tokenisation (patch_size=16 → 6×6×6 = 216 tokens)
  --------------------------------------------------------------------------------------------------------------------------
  Tokens  [B, C, p, p, p, N_patches]              (1, 4, 16, 16, 16, 216)         min=-4.866  u=-0.001  max=+4.830  std=1.000  <- each 16³ cuboid is one token, N_patches=216

  --------------------------------------------------------------------------------------------------------------------------
  1b. Intra-Modality Patch Swap (swap_prob=0.5, 2 iterations)
  --------------------------------------------------------------------------------------------------------------------------
  Swapped tokens  [B, C, p, p, p, N]              (1, 4, 16, 16, 16, 216)         min=-4.866  u=-0.001  max=+4.830  std=1.000  <- patches excha

In [4]:
# Stage 2 — SwinMamba Hybrid 3D Encoder
section("STAGE 2 │ SwinMamba Hybrid 3D Encoder")
from src.models.swin_mamba_dual import SwinMambaEncoder

enc = SwinMambaEncoder(
    in_chans=4,
    depths=(2, 2, 2, 2),
    dims=(48, 96, 192, 384),
    window_size=(7, 7, 7),
    num_heads=(3, 6, 12, 24),
).to(device).eval()

with torch.no_grad():
    # Stem
    x_stem = enc.downsample_layers[0](x_perturbed)
    log("After Stem Conv  [B, 48, 48, 48, 48]", x_stem, "halves resolution, expands 4→48 ch")

    # Stage 0 (Swin W-MSA / SW-MSA)
    x_s0 = x_stem.permute(0, 2, 3, 4, 1)
    for blk_idx, blk in enumerate(enc.swin_stages[0]):
        x_s0 = blk(x_s0)
        log(f"  Swin Stage-0 Block {blk_idx}", x_s0, "3D window attention")
    x_s0 = x_s0.permute(0, 4, 1, 2, 3).contiguous()
    x_s0 = enc.mlps[0](enc.norms[0](x_s0))
    log("Stage-0 Output [B, 48, 48, 48, 48]", x_s0, "skip feature S0")

    # Stage 1
    x_pm01 = enc.downsample_layers[1](x_s0)
    x_s1 = x_pm01.permute(0, 2, 3, 4, 1)
    for blk_idx, blk in enumerate(enc.swin_stages[1]):
        x_s1 = blk(x_s1)
    x_s1 = x_s1.permute(0, 4, 1, 2, 3).contiguous()
    x_s1 = enc.mlps[1](enc.norms[1](x_s1))
    log("Stage-1 Output [B, 96, 24, 24, 24]", x_s1, "skip feature S1")

    # Stage 2 (GSC + Mamba)
    x_pm12 = enc.downsample_layers[2](x_s1)
    x_gsc2 = enc.conv_stages[2](x_pm12)
    x_m2 = x_gsc2
    for layer_idx, mamba_layer in enumerate(enc.mamba_stages[2]):
        x_m2 = mamba_layer(x_m2)
    x_s2 = enc.mlps[2](enc.norms[2](x_m2))
    log("Stage-2 Output [B, 192, 12, 12, 12]", x_s2, "tri-directional Mamba skip S2")

    # Stage 3
    x_pm23 = enc.downsample_layers[3](x_s2)
    x_gsc3 = enc.conv_stages[3](x_pm23)
    x_m3 = x_gsc3
    for layer_idx, mamba_layer in enumerate(enc.mamba_stages[3]):
        x_m3 = mamba_layer(x_m3)
    x_s3 = enc.mlps[3](enc.norms[3](x_m3))
    log("Stage-3 Output [B, 384, 6, 6, 6]", x_s3, "deepest bottleneck representation S3")


  STAGE 2 │ SwinMamba Hybrid 3D Encoder
  After Stem Conv  [B, 48, 48, 48, 48]            (1, 48, 48, 48, 48)             min=-0.029  u=+0.132  max=+2.799  std=0.250  <- halves resolution, expands 4→48 ch
    Swin Stage-0 Block 0                          (1, 48, 48, 48, 48)             min=-1.235  u=+0.164  max=+2.936  std=0.338  <- 3D window attention
    Swin Stage-0 Block 1                          (1, 48, 48, 48, 48)             min=-1.629  u=+0.126  max=+3.059  std=0.442  <- 3D window attention
  Stage-0 Output [B, 48, 48, 48, 48]              (1, 48, 48, 48, 48)             min=-0.571  u=-0.003  max=+0.625  std=0.103  <- skip feature S0
  Stage-1 Output [B, 96, 24, 24, 24]              (1, 96, 24, 24, 24)             min=-0.277  u=-0.006  max=+0.263  std=0.082  <- skip feature S1
  Stage-2 Output [B, 192, 12, 12, 12]             (1, 192, 12, 12, 12)            min=-0.755  u=+0.001  max=+0.731  std=0.135  <- tri-directional Mamba skip S2
  Stage-3 Output [B, 384, 6, 6, 6]        

In [5]:
# Stage 3 — Skip Connection Encoders & Bottleneck
section("STAGE 3 │ Skip Connection Encoders + Bottleneck")
from src.models.unetr_blocks import UnetrBasicBlock3D

enc_stem_blk   = UnetrBasicBlock3D(in_channels=4,   out_channels=48).eval()
enc_s0_blk     = UnetrBasicBlock3D(in_channels=48,  out_channels=48).eval()
enc_s1_blk     = UnetrBasicBlock3D(in_channels=96,  out_channels=96).eval()
enc_s2_blk     = UnetrBasicBlock3D(in_channels=192, out_channels=192).eval()
enc_bottle_blk = UnetrBasicBlock3D(in_channels=384, out_channels=768).eval()

with torch.no_grad():
    skip_stem  = enc_stem_blk(x_perturbed)
    skip_0     = enc_s0_blk(x_s0)
    skip_1     = enc_s1_blk(x_s1)
    skip_2     = enc_s2_blk(x_s2)
    bottleneck = enc_bottle_blk(x_s3)

    log("skip_stem   [B, 48, 96, 96, 96]", skip_stem, "raw input skip")
    log("skip_0      [B, 48, 48, 48, 48]", skip_0, "Stage 0 skip")
    log("skip_1      [B, 96, 24, 24, 24]", skip_1, "Stage 1 skip")
    log("skip_2      [B, 192, 12, 12, 12]", skip_2, "Stage 2 skip")
    log("bottleneck  [B, 768, 6, 6, 6]", bottleneck, "bottleneck features 384→768")


  STAGE 3 │ Skip Connection Encoders + Bottleneck
  skip_stem   [B, 48, 96, 96, 96]                 (1, 48, 96, 96, 96)             min=-0.084  u=+0.406  max=+8.514  std=0.655  <- raw input skip
  skip_0      [B, 48, 48, 48, 48]                 (1, 48, 48, 48, 48)             min=-0.061  u=+0.385  max=+6.038  std=0.588  <- Stage 0 skip
  skip_1      [B, 96, 24, 24, 24]                 (1, 96, 24, 24, 24)             min=-0.060  u=+0.384  max=+5.697  std=0.598  <- Stage 1 skip
  skip_2      [B, 192, 12, 12, 12]                (1, 192, 12, 12, 12)            min=-0.051  u=+0.399  max=+5.571  std=0.594  <- Stage 2 skip
  bottleneck  [B, 768, 6, 6, 6]                   (1, 768, 6, 6, 6)               min=-0.053  u=+0.393  max=+4.864  std=0.591  <- bottleneck features 384→768


In [6]:
# Stage 4 — UNETR Decoder with EV2 Skip Fusion
section("STAGE 4 │ UNETR Progressive Decoder (EV2 Skip Fusion)")
from src.models.unetr_blocks import UnetrUpBlock3D, EV2Block

dec4 = UnetrUpBlock3D(in_channels=768, out_channels=192).eval()
dec3 = UnetrUpBlock3D(in_channels=192, out_channels=96).eval()
dec2 = UnetrUpBlock3D(in_channels=96,  out_channels=48).eval()
dec1 = UnetrUpBlock3D(in_channels=48,  out_channels=48).eval()
refine = EV2Block(in_channels=48, out_channels=48).eval()

with torch.no_grad():
    d4 = dec4(bottleneck, skip_2)
    log("dec4 (bottleneck↑ ⊕ skip_2) [B, 192, 12, 12, 12]", d4, "6³→12³ upsampling")
    d3 = dec3(d4, skip_1)
    log("dec3 (d4↑ ⊕ skip_1)         [B, 96, 24, 24, 24]", d3, "12³→24³ upsampling")
    d2 = dec2(d3, skip_0)
    log("dec2 (d3↑ ⊕ skip_0)         [B, 48, 48, 48, 48]", d2, "24³→48³ upsampling")
    d1 = dec1(d2, skip_stem)
    log("dec1 (d2↑ ⊕ skip_stem)      [B, 48, 96, 96, 96]", d1, "48³→96³ upsampling")
    dec_out = refine(d1)
    log("EV2 Refinement (SE Attn)   [B, 48, 96, 96, 96]", dec_out, "final decoder features")


  STAGE 4 │ UNETR Progressive Decoder (EV2 Skip Fusion)
  dec4 (bottleneck↑ ⊕ skip_2) [B, 192, 12, 12, 12]  (1, 192, 12, 12, 12)            min=-0.060  u=+0.596  max=+5.431  std=0.665  <- 6³→12³ upsampling
  dec3 (d4↑ ⊕ skip_1)         [B, 96, 24, 24, 24]  (1, 96, 24, 24, 24)             min=-0.061  u=+0.592  max=+6.666  std=0.672  <- 12³→24³ upsampling
  dec2 (d3↑ ⊕ skip_0)         [B, 48, 48, 48, 48]  (1, 48, 48, 48, 48)             min=-0.070  u=+0.589  max=+7.128  std=0.676  <- 24³→48³ upsampling
  dec1 (d2↑ ⊕ skip_stem)      [B, 48, 96, 96, 96]  (1, 48, 96, 96, 96)             min=-0.080  u=+0.572  max=+8.669  std=0.698  <- 48³→96³ upsampling
  EV2 Refinement (SE Attn)   [B, 48, 96, 96, 96]  (1, 48, 96, 96, 96)             min=-0.082  u=+0.781  max=+9.951  std=0.805  <- final decoder features


In [7]:
# Stage 5 — Output Heads (Segmentation, Classification, Reconstruction)
section("STAGE 5 │ Task Output Heads")
from src.models.unetr_blocks import UnetOutBlock3D
from src.models.swin_mamba_dual import ClassificationHead
import torch.nn.functional as F

seg_head   = UnetOutBlock3D(in_channels=48, out_channels=3).eval()
clf_head   = ClassificationHead(in_features=768, num_classes=2, dropout=0.4).eval()
recon_head = torch.nn.Conv3d(48, 4, kernel_size=3, stride=1, padding=1).eval()

with torch.no_grad():
    # 5a. Segmentation
    seg_logits = seg_head(dec_out)
    seg_probs  = torch.sigmoid(seg_logits)
    log("5a. seg_logits [B, 3, 96, 96, 96]", seg_logits, "channels: [0]=TC [1]=WT [2]=ET")
    log("    seg_probs  [B, 3, 96, 96, 96]", seg_probs, "sigmoid probabilities")

    # 5b. Classification
    clf_logits = clf_head(bottleneck)
    clf_probs  = F.softmax(clf_logits, dim=-1)
    log("5b. clf_logits [B, 2]", clf_logits, "[0]=LGG [1]=HGG")
    log("    clf_probs  [B, 2]", clf_probs, f"LGG={clf_probs[0,0]:.4f} HGG={clf_probs[0,1]:.4f}")

    # 5c. Reconstruction (Pretraining)
    recon_out = recon_head(dec_out)
    log("5c. recon_out  [B, 4, 96, 96, 96]", recon_out, "reconstructed 4-modality volume")

print(_DIVIDER)
print("  Module-Wise Verification Complete!")
print(_DIVIDER)


  STAGE 5 │ Task Output Heads
  5a. seg_logits [B, 3, 96, 96, 96]               (1, 3, 96, 96, 96)              min=-2.885  u=-0.008  max=+3.318  std=0.492  <- channels: [0]=TC [1]=WT [2]=ET
      seg_probs  [B, 3, 96, 96, 96]               (1, 3, 96, 96, 96)              min=+0.053  u=+0.497  max=+0.965  std=0.114  <- sigmoid probabilities
  5b. clf_logits [B, 2]                           (1, 2)                          min=-0.515  u=-0.200  max=+0.115  std=0.446  <- [0]=LGG [1]=HGG
      clf_probs  [B, 2]                           (1, 2)                          min=+0.347  u=+0.500  max=+0.653  std=0.216  <- LGG=0.3474 HGG=0.6526
  5c. recon_out  [B, 4, 96, 96, 96]               (1, 4, 96, 96, 96)              min=-3.557  u=+0.256  max=+4.405  std=0.783  <- reconstructed 4-modality volume
  Module-Wise Verification Complete!
